In [0]:
%pip install python-Levenshtein biotite gemmi py2Dmol

In [0]:
%restart_python

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

### Combine Metrics across all Protein-Hunter Runs


In [0]:
import os
import pandas as pd
def merge_metrics(overarching_folder_path: str) -> pd.DataFrame:
    """ Merges the metrics from the 2 different metric assessment files into a single dataframe.
            - Apo Boltz Metrics
            - Holo AF2 & Boltz Metrics
            - Physics Metrics
    Args:
        overarching_folder_path (str): Path to the folder containing the metrics files.
        
    Returns:
        pd.DataFrame: Merged metrics dataframe
    """
    path_final_metrics = os.path.join(overarching_folder_path, "final_metrics")
    for filename in os.listdir(path_final_metrics):
        
        if "physics" in filename:
            path_physics_metrics = os.path.join(path_final_metrics, filename)
            df_physics_metrics = pd.read_csv(path_physics_metrics, index_col= 0)
            df_physics_final = df_physics_metrics.copy()
            df_physics_final['design_name'] = df_physics_final['pdb_filename'].str.split('_model_').str[0]
        
        elif "apo" in filename:
            path_apo_metrics = os.path.join(path_final_metrics, filename)
            df_apo_metrics = pd.read_csv(path_apo_metrics)
            agg_funcs = ['mean', 'median', 'std']
            df_apo_agg = df_apo_metrics.groupby('design_name').agg({
                                                            "ptm" : agg_funcs,
                                                            "complex_plddt" : agg_funcs,
                                                            })
            df_apo_agg.reset_index(inplace = True)
            df_apo_agg.columns = [("_".join(col)).strip("_") for col in df_apo_agg.columns]
            df_apo_model0 = df_apo_metrics[df_apo_metrics['model_id'] == 0].copy()
            if 'path_boltz_apo_structure' not in df_apo_model0.columns:
                df_apo_model0['path_boltz_apo_structure'] = df_apo_model0['path_apo_boltz_structure']
            df_apo_agg = pd.merge(df_apo_agg, df_apo_model0[['design_name', 'path_boltz_apo_structure']], on = 'design_name', how = 'left')
            df_apo_final = df_apo_agg.rename(columns = {'ptm_mean': 'ptm_apo', 'complex_plddt_mean': 'plddt_apo'})
        
        elif "holo" in filename:
            path_holo_metrics = os.path.join(path_final_metrics, filename)
            df_holo_metrics = pd.read_csv(path_holo_metrics, index_col = 0)
            weakly_correlated_cols = ['ligand_iptm_mean', 'complex_plddt', 'complex_iplddt', 'iptm', 'ptm_boltz', 'ligand_iptm', 'rmsd_holo_binder_rfd_boltz',
                          'plddt', 'plddt_binder', 'binder_chain', 'target_chain', 'rmsd']
            drop_contact_cols = [x for x in df_holo_metrics.columns if ("paratope" in x) or ("epitope" in x)]
            full_drop_cols = weakly_correlated_cols + drop_contact_cols
            df_holo_copy = df_holo_metrics.drop(columns = full_drop_cols)
            df_holo_final = df_holo_copy.rename(columns = {'complex_plddt_mean' : 'plddt', 'complex_iplddt_mean' : 'iplddt', 'complex_pde_mean' : 'pde', 
                               'ptm_boltz_mean' : 'ptm_boltz', 'rmsd_holo_binder_rfd_boltz_mean' : 'rmsd_holo_binder_rfd_boltz',
                               'seq_binder' : 'sequence', 'seq' : 'seq_target_binder', 'protein_iptm_mean' : 'iptm'})
    
    # Merge across all 3 different validation steps
    df_metrics_pt1 = pd.merge(left = df_holo_final, right = df_apo_final, on = 'design_name', how = 'inner')
    df_metrics = pd.merge(left = df_metrics_pt1, right = df_physics_final, on = 'design_name', how = 'inner')

    return df_metrics
    
#----------Use if trying to merge metrics across multiple distinct ProteinHunter runs
def merge_metrics_across_runs(overarching_folder_paths: list) -> pd.DataFrame:
    """ Merges metrics across multiple distinct ProteinHunter Runs"""
    metrics_across_runs = []
    for overarching_folder_path in overarching_folder_paths:
        df_metrics = merge_metrics(overarching_folder_path)
        print("Number of Designs in Run: ", df_metrics.shape[0])
        print("Number of metrics: ", len(df_metrics.columns))
        print("-----------------------------------------------------------")
        metrics_across_runs.append(df_metrics)
    
    df_final = pd.concat(metrics_across_runs, axis = 0).reset_index(drop = True)
    if df_final['sequence'].is_unique == False:
        df_final = df_final.drop_duplicates(subset = ['sequence'], keep = 'first')
        print("Designed Sequences are not unique")
    print("Number of Designs in Combined Runs: ", df_final.shape[0])
    return df_final

path_design_campaigns = ["/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular/",
                        "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial2/",
                        "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial3/"]
overarching_folder_paths = [f"{path_design_campaign}/rfdiffusion" for path_design_campaign in path_design_campaigns]
df_metrics = merge_metrics_across_runs(overarching_folder_paths)
df_metrics

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots()
data_col = 'surface_hydrophobicity'
sns.histplot(x = data_col, data = df_metrics, ax = ax)
ax.set_xlabel(data_col);
ax.set_title(f"Distribution of {data_col}");

In [0]:
print("Data Metric: ", data_col)
df_metrics[data_col].describe()

### Adding RMSD Metrics to Final Ranking

In [0]:
rmsd_threshold_weight = {
    'neg_rmsd_holo_binder_rfd_boltz' : {'threshold' : -2,  'weight' : 2},
    'neg_rmsd_holo_binder_rfd_af2' :   {'threshold' : -2,  'weight' : 2},
    'neg_interface_holo_apo_rmsd' :    {'threshold' : -1.5,  'weight' : 1.5}, 
    'neg_rmsd_motif' :                 {'threshold': -1.5, 'weight' : 1},
    'neg_rmsd_middle_strand' :         {'threshold' : -1.5, 'weight' : 1},
}
rmsd_cols = [x for x in df_metrics.columns if (('rmsd' in x) & ('median' not in x) & ('std' not in x))]
rmsd_cols

In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/Scfv_Toolkit")
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics") 
from RankDesign import RankDesign
rank_design_obj = RankDesign()

for rmsd_col in rmsd_cols:
    neg_rmsd_col = f"neg_{rmsd_col}"
    threshold = rmsd_threshold_weight[neg_rmsd_col]['threshold']
    weight = rmsd_threshold_weight[neg_rmsd_col]['weight']

    # Add new metric to ranking scheme. Specifically, add the metric name, weight (importance in ranking), threshold for success, and whether to flip
    rank_design_obj.update_threshold(metric = neg_rmsd_col, threshold = threshold)
    rank_design_obj.update_weight(metric = neg_rmsd_col, weight = weight)
    rank_design_obj.update_cols_to_flip(metric = rmsd_col)

In [0]:
from StrucTools import *
from RFDUtils import *

list_sequence_designed = []
for index, row in df_metrics.iterrows():
    
    contig = df_metrics['contig'].iloc[index]
    filepath_apo = df_metrics['path_boltz_apo_structure'].iloc[index]
    sequence = df_metrics['sequence'].iloc[index]

    # 1. Extract Binder Contig
    print("Contig: ", contig)
    binder_contig = extract_binder_contig(contig)
    print("Binder_contigs: ", binder_contig)

    # 2. Create mask aligning to motif with True and non-motif or design = False
    example_coords, align_mask = extract_design_motif_coords(filepath_apo, binder_contig, binder_chain_id= 'A', 
                                                             return_coords_mask = True, desired_motif_chain_id= "")
    #3. Create design_mask
    design_mask = ~np.array(align_mask)
    sequence_designed = "".join([sequence[index] for index, val in enumerate(design_mask) if val == True])
    list_sequence_designed.append(sequence_designed)

df_metrics['sequence_designed'] = list_sequence_designed
df_metrics

### Run Entire Filter and Ranking Pipeline

In [0]:
top_n = 23
alpha = 0.1
df_top_designs = rank_design_obj.run_filter_rank_pipeline(df_designs = df_metrics, 
                                                          design_mask = design_mask, 
                                                          top_n = top_n,
                                                          alpha = alpha)
df_top_designs.reset_index(drop = True, inplace = True)
df_top_designs
                                                

In [0]:
list_sequence_motif = []
for index, row in df_top_designs.iterrows():
    
    contig = df_top_designs['contig'].iloc[index]
    filepath_apo = df_top_designs['path_boltz_apo_structure'].iloc[index]
    sequence = df_top_designs['sequence'].iloc[index]

    # 1. Extract Binder Contig
    print("Contig: ", contig)
    binder_contig = extract_binder_contig(contig)
    print("Binder_contigs: ", binder_contig)

    # 2. Create mask aligning to motif with True and non-motif or design = False
    example_coords, motif_mask = extract_design_motif_coords(filepath_apo, binder_contig, binder_chain_id= 'A', 
                                                             return_coords_mask = True, desired_motif_chain_id= "")
    #3. Create design_mask
    sequence_motif = "".join([sequence[index] for index, val in enumerate(motif_mask) if val == True])
    list_sequence_motif.append(sequence_motif)

assert(len(np.unique(list_sequence_motif)) == 1)

In [0]:
for path in df_top_designs['path_holo_boltz_structure'].values:
    visualize_structure(path)

Concerned about super long stable alpha helix used as prefix to design and deciding to include shortened version as a positive control


Specifically based on design with iloc # 17

In [0]:
index_of_interest = 17
print("Sequence of Target: ")
print(df_top_designs.iloc[index_of_interest]['seq_target'])
print("Sequence of Binder: ")
print(df_top_designs.iloc[index_of_interest]['sequence'][25:])

In [0]:
temp_row = df_top_designs.iloc[17]
temp_row['sequence'] = df_top_designs.iloc[index_of_interest]['sequence'][25:]
df_top_designs_final = pd.concat([df_top_designs, temp_row.to_frame().T], ignore_index=True, axis = 0)

In [0]:
import biotite.sequence as seq
tag_seq = "MLEHHHHHH"
tag_obj = seq.ProteinSequence(sequence = tag_seq)
tag_weight = tag_obj.get_molecular_weight()
print("Tag Weight: ", tag_weight)
df_top_designs_final['mol_weight'] = df_top_designs_final['sequence'].apply(lambda sequence: seq.ProteinSequence(sequence = sequence).get_molecular_weight()) + tag_weight

# Add Tag to sequence for copying over to Smartsheets
df_top_designs_final['sequence_with_tags'] = "M" + df_top_designs_final['sequence'] + "LEHHHHHH"
# Add Simpler Name for SmartSheet and Twist
df_top_designs_final['design_name_simple'] = [f"dbl_strnd_collagen_rnd1_rank{index+1}" for index in range(len(df_top_designs_final))]
df_top_designs_final['mol_weight']

In [0]:
df_top_designs_final.iloc[17][['plddt', 'iplddt', 'ptm_apo', 'plddt_apo', 'iptm', 'ptm_boltz']]

In [0]:
df_top_designs_final['sequence'].str.len()

In [0]:
# removed 23. Keep in mind Twist has a 100 amino acid minimum design limit. Seqs for the ordered designs with less than 100 may have LEHHHHH added to them to compensate. review the sequence to make sure

In [0]:
df_top_designs_final.to_csv("/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_ordered_designs/april_24_dble_strand_collagen_ordered_designs.csv")